# 02. Treinamento do Modelo (Classificação Binária)

Este notebook orquestra o treinamento da DenseNet161 para classificar imagens em **Pneumonia (0) ou COVID-19 (1)**.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import wandb
from pytorch_lightning.loggers import WandbLogger
from pytorch_lightning import Trainer
from pytorch_lightning.callbacks import ModelCheckpoint, LearningRateMonitor
import torch

from src.config import Config
from src.model import SimpleClassifier
from dataset.loaders import train_loader, val_loader

### Login no Weights & Biases
A chave da API já deve ter sido passada pelo `docker-compose.yml`. Caso contrário, ele pedirá para você colar o token abaixo.

In [ ]:
wandb.login()

### Inicialização do Modelo e Logger

In [ ]:
# Inicializa o Logger do Wandb
wandb_logger = WandbLogger(
    project="SIATCT_Binary_Experiment",
    name=f"DenseNet_{Config.EXPERIMENT_NAME}",
    log_model="all"
)

# Instancia o modelo
model = SimpleClassifier(
    num_classes=Config.NUM_CLASSES, 
    learning_rate=Config.LEARNING_RATE,
    weight_decay=Config.WEIGHT_DECAY
)

### Configuração de Callbacks e Treinamento

In [ ]:
# Checkpoint para salvar os melhores pesos
checkpoint_callback = ModelCheckpoint(
    dirpath=Config.CHECKPOINTS_DIR,
    filename="{epoch}-{val_loss:.2f}-{val_acc:.2f}",
    monitor="val_loss",
    mode="min",
    save_top_k=3,
    save_last=True,
)

lr_monitor = LearningRateMonitor(logging_interval="epoch")

trainer = Trainer(
    max_epochs=Config.MAX_EPOCHS,
    logger=wandb_logger,
    callbacks=[checkpoint_callback, lr_monitor],
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    devices=1,
    precision="16-mixed", # Treinamento em precisão mista para acelerar e salvar memória
    limit_train_batches=0.1,  # <--- Roda apenas 10% do conjunto de treino por época
    limit_val_batches=0.1,    # <--- Roda apenas 10% do conjunto de validação por época
)

In [ ]:
# Inicia o treinamento
trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=val_loader)

# Finaliza o run do wandb
wandb.finish()